# Stage 0 — Hypothesis v2: Stretched Volume Profile Fade

**Date locked:** 2026-05-24  
**Author:** Anthony Chung  
**Version:** v2.0  
**Predecessor:** v1.0 Asian Session Extreme Fade (FAILED Stage 2: 0/1500 trials profitable)  
**Status:** ACTIVE  
**Symbol scope (Phase 1):** BTCUSDT  
**Timeframe:** 1h  
**Source:** Axia Futures YT video `MUv2GfGAW_I` — "How To Trade The Stretched Volume Profile Strategy In Crypto"

---

## Why v2

v1 (Asian session extreme fade) failed at Stage 2: 0 of 1,500 IS configurations produced positive Sharpe, with best PF = 0.76. Conclusion: the v1 thesis (thin-liquidity overreaction at session boundary) does not produce an edge on BTC H1 large enough to overcome HORROR-mode costs.

v2 targets a DIFFERENT inefficiency — **volume-profile dislocation** — derived from Axia Futures' published methodology (Brannigan Barrett, head trader). The underlying mechanism is similar in spirit (mean reversion) but uses a fundamentally different signal: where the *value area* was during the prior session, not just the price extremes.

Per SOP, this is a versioned hypothesis. No parameter from v1 is reused; the entire test budget is fresh.

## Thesis

At UTC 23:00 each day, compute that day's Volume Profile. If the day's close has "stretched" — i.e., trades meaningfully **above** the Value Area High (VAH) or **below** the Value Area Low (VAL) — the next session is more likely than not to mean-revert toward the Point of Control (POC).

## Market Inefficiency Targeted

**Liquidity-dislocation reversion**:

1. The day's value area (VA, 70% of volume) represents the price range where genuine two-way consensus was reached during the session.
2. A close significantly outside the VA implies the final-hour buyer (or seller) paid a price not validated by the day's deeper liquidity — typically a stop-out, panic, or flow-driven event.
3. Fresh liquidity arriving in the next session has no commitment to defend that stretched price and is more likely to fade it.
4. The POC acts as the natural target — it is the most-traded price and therefore the deepest liquidity anchor.

This is distinct from a generic mean-reversion thesis because the *reference price* is not a moving average — it is a structurally-defined fair-value point derived from realized volume.

## Theoretical and Empirical Support

| Reference | Contribution |
|-----------|--------------|
| Steidlmayer (1986), *Markets and Market Logic* | Originator of Market Profile / Value Area framework |
| Brannigan Barrett — Axia Futures (2021 video) | Crypto-specific demonstration of stretched-VP fade |
| Madhavan, Richardson & Roomans (1997) | Empirical evidence on intraday liquidity dynamics |
| Hasbrouck (1991), *Measuring the Information Content of Stock Trades* | Theoretical framework for permanent vs transient price impact |

The Volume Profile community in crypto is small but active; this strategy is a clean systematization of a discretionary technique used by professional prop traders.

## Strategy Specification

### Volume Profile Computation (per UTC day)
- Aggregate the 24 H1 bars of each day.
- Distribute each bar's volume **linearly** across its [low, high] range.
- Bucket into 50 equal-width price buckets between day-low and day-high.
- **POC** = bucket with max volume.
- **Value Area** = expand from POC until cumulative volume ≥ 70% of total.
- **VAH** = upper edge of VA, **VAL** = lower edge.

### Entry (evaluated at UTC 23:00 close of day D)
1. **Range filter**: `(day_high - day_low) / day_open` must be ≥ `min_range_pct` (avoids dead days).
2. **Stretched condition**:
   - SHORT if `close > VAH × (1 + stretched_threshold_pct)`
   - LONG  if `close < VAL × (1 − stretched_threshold_pct)`
3. Entry executes at **open of UTC 00:00 of day D+1** (first bar of next day).

### Exit Rules (priority order)
1. **TP**: POC of day D (the fade target).
2. **SL**: day-D high × (1 + buffer) for SHORTS; day-D low × (1 − buffer) for LONGS.
3. **Time stop**: forced flat after `time_stop_hours` from entry.

### Position Sizing
- Fixed-fractional 1% equity risk via stop distance.
- Max 1 concurrent position.

### Costs
- HORROR mode (0.20% slip + 0.12% taker). Round-trip ≈ 0.64%.

## Locked Parameter Search Space

| Parameter | Search values | Notes |
|-----------|---------------|-------|
| `min_range_pct` | [0.005, 0.010, 0.015, 0.020, 0.025] | filter weak days |
| `stretched_threshold_pct` | [0.005, 0.010, 0.015, 0.020, 0.025] | how far past VAH/VAL to trigger |
| `stop_buffer_pct` | [0.005, 0.010, 0.015, 0.020] | SL distance beyond extreme |
| `time_stop_hours` | [12, 24, 48] | hold horizon |
| `va_pct` | 0.70 (locked) | Steidlmayer standard |
| `n_buckets` | 50 (locked) | resolution trade-off |

**Total trials:** 5 × 5 × 4 × 3 = **300**

N_trials recorded as 300 for Stage 5 Deflated Sharpe correction.

## Falsifiable Predictions

| # | Test | Threshold | Stage |
|---|------|-----------|-------|
| 1 | At least 1 of 300 IS trials has PF > 1.0 | required to proceed | 2 |
| 2 | IS → OOS-1 PF retention | ≥ 70% | 3 |
| 3 | Walk-forward profitable months | ≥ 80% | 4 |
| 4 | Holdout PF | > 1.3 | 5 |
| 5 | Holdout Sharpe | > 1.0 | 5 |
| 6 | Holdout MaxDD | < 30% | 5 |
| 7 | Deflated Sharpe (N=300) | > 0 | 5 |
| 8 | Multi-coin replication | ≥ 4 of 6 pass tests 4-6 | 6 |

Stage 2 gate added for this hypothesis: if even with HORROR costs **zero** trials are profitable, the thesis is dead at Stage 2 (no further work).

## Data Scope

Identical to v1 — same split is reused (data does not depend on the strategy):

| Aspect | Value |
|--------|-------|
| Symbol (Phase 1) | BTCUSDT |
| Timeframe | 1h |
| Date range | 2022-01-01 to 2026-04-01 |
| IS / OOS-1 / Holdout | 60% / 20% / 20% (time-ordered) |
| Stage 6 symbols | ETHUSDT, SOLUSDT, AVAXUSDT, XRPUSDT, BNBUSDT |

## Pre-Commitment Statement

> I have not examined the backtest of this v2 specification prior to this commit. The parameter ranges are a-priori judgments based on standard Volume Profile practice and reasonable crypto price-scale considerations.
>
> The Stretched VP strategy uses **completely independent** parameters from v1; no v1 result has been used to tune v2.
>
> If v2 fails, a v3 hypothesis (different mechanism) requires a new notebook. Repeated tweaking of v2 parameters after seeing test-set results is forbidden.
>
> — Anthony Chung, 2026-05-24